# Lab 2（上）財務資料清理與分析：把爬到的髒表格洗乾淨、看懂它

上一個 Lab 你把台積電股價爬成了 `prices.csv`，但那份資料是「髒」的。
今天先把它洗乾淨，再學會用幾行程式看懂一整個月的行情。

In [ ]:
# 📦 先跑這一格：指定版本，避免學校電腦裝到不相容的舊版
!pip install -q pandas==2.2.3 numpy==1.26.4 matplotlib==3.10.0

In [ ]:
# 老師的小設定：載入今天整本要用的工具、設好中文字型（直接跑、不用改）
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

# 讓圖上的中文正常顯示（清單第一個是本機認得的字型，其餘是不同系統的備援）
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Noto Sans CJK TC", "Microsoft JhengHei", "PingFang TC", "AR PL UMing CN", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
print("工具載入完成")

## 🔧 第 0 步：環境檢查

In [ ]:
import pandas as pd
import matplotlib
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("環境 OK，可以開始")

## A・修 `prices.csv`：上一個 Lab 留下的兩個髒點

上一個 Lab 爬完股價，當場指出兩個地方沒修：日期是民國年、數字帶逗號。今天就修它們。

### A1・照妖鏡：看每一欄到底是什麼型別

> 預期：`日期`、`成交股數`、`收盤價` 等欄位型別全是 `object`（＝文字），不是數字。

In [ ]:
df = pd.________("data/prices.csv")   # TODO: 用 pandas 讀 CSV 檔的函式
print(df._______)                      # TODO: 每一欄的「型別」屬性（是屬性、不加括號）—— 長得像數字≠真的是數字
df.head()                              # 💡 最後一行不用 print，Jupyter 會直接把表格畫出來

### A2・先嚐痛點：叫文字欄算平均會怎樣

> ⚠️ 這一格會**故意報錯**（紅字），這是正常的、預期中的——看完往下走就好。

In [ ]:
df["成交股數"].______()      # TODO: 算平均的方法（這一欄其實是文字，會故意報錯，正常）
# 💡 這個錯不是 bug，是 pandas 在保護你：它不猜你想怎麼轉，要你先講清楚

### A3・修第一個髒點：民國日期轉西元

> 預期：印出 `2026-06-01 00:00:00`、型別從 `object` 變成 `datetime64`。

In [ ]:
def roc_to_date(s):                       # '115/06/01' → 2026-06-01
    y, m, d = s.split("/")                # 💡 拆成 ['115', '06', '01']
    return pd.Timestamp(int(y) + 1911, int(m), int(d))   # 💡 民國 + 1911 = 西元

df["日期"] = df["日期"].________(roc_to_date)   # TODO: 對整欄「每一格都套用同一個函式」的方法
print(df["日期"].iloc[0])
print(df["日期"].dtype)                       # 💡 變成 datetime64 → 之後排序、畫圖才會照時間

### A4・修第二個髒點：拔逗號 + 轉成數字

> 預期：`成交股數` 變 `int64`、`收盤價` 等變 `float64`。

In [ ]:
def to_int(s):                            # '60,942,792' → 60942792
    return int(s.replace(",", ""))        # 💡 先拔逗號、再轉整數——順序反了會報錯

def to_float(s):                          # '2,355.00' → 2355.0
    return float(s.replace(",", ""))

for col in ["成交股數", "成交金額", "成交筆數"]:
    df[col] = df[col].apply(_______)       # TODO: 上面定義的兩個函式，哪個把「字串→整數」？

for col in ["開盤價", "最高價", "最低價", "收盤價"]:
    df[col] = df[col].apply(_______)     # TODO: 哪個把「字串→小數」？

print(df.dtypes)

### A5・驗收：剛剛報錯的那行，現在算得出來了

> 預期：印出一個平均股數（不再報錯）。

In [ ]:
print("這個月平均每天成交股數：", round(df["成交股數"].mean()))
# 💡 同一份資料，洗乾淨之前連平均都算不了——這就是資料清理的價值

## B・敘述統計：一行看懂整個月的行情

資料乾淨了，先別急著建模——用幾行程式先「看懂」它。

### B1・一行看懂每一欄：摘要統計

> 預期：一張表，每欄印出筆數/平均/標準差/最小/四分位/最大。

In [ ]:
df[["開盤價", "最高價", "最低價", "收盤價"]]._______()   # TODO: 一次給你八個摘要數字（筆數/平均/標準差/最大最小…）的方法
# 💡 最後一行不用 print，Jupyter 直接畫表

### B2・挑幾個最常看的數字自己算一次

> 預期：印出收盤價的平均、最高、最低、波動。

In [ ]:
close = df["收盤價"]                       # 💡 先把收盤價這一欄取出來，等一下重複用
print("平均收盤價：", round(close._______(), 2))          # TODO: 算平均
print("最高：", close._______(), "  最低：", close._______())   # TODO: 最大／最小
print("波動（標準差）：", round(close._______(), 2))   # TODO: 標準差＝這個月跳動的劇烈程度

## C・畫圖：看圖比看數字快

同樣一份資料，畫成圖，趨勢一眼就看出來。

### C1・收盤價走勢：折線圖

> 預期：一張折線圖，X 軸是日期、Y 軸是收盤價。

In [ ]:
df = df.sort_values("日期")               # 💡 畫圖前先按日期排好，線才不會亂跳

plt.figure(figsize=(9, 4))                # 💡 開一張圖，設定大小
plt._______(df["日期"], df["收盤價"], marker="o")   # TODO: 畫折線圖的函式；marker='o' 每個交易日點一個點
plt.title("台積電 2330 這個月收盤價走勢")
plt.xlabel("日期")
plt.ylabel("收盤價（元）")
plt.xticks(rotation=45)                   # 💡 日期標籤轉 45 度，才不會擠在一起
plt.tight_layout()
plt.show()

### C2・每天成交量：長條圖

> 預期：一張長條圖，看得出哪幾天量特別大。

In [ ]:
plt.figure(figsize=(9, 4))
plt._______(df["日期"], df["成交股數"])       # TODO: 畫長條圖的函式，適合看「每天各多少」
plt.title("台積電 2330 這個月每日成交股數")
plt.xlabel("日期")
plt.ylabel("成交股數")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 🔑 這一段你做了什麼

1. 把爬到的髒資料修好（民國轉西元、拔逗號轉數字）
2. 用摘要統計和幾個數字看懂整月行情
3. 把資料畫成折線圖、長條圖

下一段（Lab 2 下）：換一個更方便的資料來源，一行就拿到好幾年的股價。